### Azure AI Search Multi-Modal RAG - Simple Image Verbalisation Lab

![simple_image_verbalisation.png](./Assets/simple_image_verbalisation.png)

### Installing Required Libraries and Utilities

In [ ]:
%pip install azure-ai-projects python-dotenv openai requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### Setting Up the Environment

In [10]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

AGENT_ENDPOINT = os.getenv("AGENT_ENDPOINT")
AGENT_NAME = os.getenv("AGENT_NAME")

credentials = DefaultAzureCredential() 

### Creating the Agent Client

In [12]:
project_client = AIProjectClient(
    endpoint=AGENT_ENDPOINT,
    credential=credentials
)

### Bind the OpenAI client to the existing agent endpoint.

In [13]:
openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)

### Call the agent 

In [14]:
conversation = openai_client.conversations.create()
user_message = "can you tell me something about the gender equality in BMW?"
response = openai_client.responses.create(
    conversation=conversation.id,
    input=[{"role": "user", "content": user_message}],
)

print(f"Response output: {response.output_text}")

Response output: BMW actively promotes gender equality and diversity through structured initiatives and strategies. It sees diversity as an asset, addressing five main dimensions: gender, age, cultural background, sexual orientation, and physical/mental abilities, as part of its inclusion efforts. In 2024, the company had 127,317 male employees and 31,080 female employees, with women making up 2,752 out of the 12,755 individuals in top management【3:0†source】【3:2†source】.

To bridge pay gaps, BMW reports an unadjusted gender pay gap of -10.9% and focuses on employee development and opportunities, investing in training and inclusion in recruitment【3:0†source】【3:2†source】. BMW also fosters awareness through training and discussions to embed the value of diversity at every level【3:2†source】.


## Perform the operation manually  

### Set up the environment

In [2]:
service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
key = os.getenv("AZURE_SEARCH_API_KEY")

openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_key = os.getenv("AZURE_OPENAI_KEY")
openai_embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

### Create the openai client   

In [3]:
from openai import AzureOpenAI


azure_openai_client = AzureOpenAI(
    api_key=openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=openai_endpoint
)

### Creating the Embedding Generator Function

In [4]:
def generate_embeddings(client: AzureOpenAI, text):
    embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']

In [5]:
user_query = "can you tell me something about the sustainability initiatives at BMW?"
vectorised_user_query = generate_embeddings(azure_openai_client, user_query)
print(vectorised_user_query)

[0.01983642578125, 0.004840850830078125, -0.0245208740234375, 0.01483917236328125, -0.00988006591796875, 0.003070831298828125, 0.011962890625, 0.018890380859375, -0.004482269287109375, -0.0052947998046875, 0.0256195068359375, -0.03338623046875, 0.01523590087890625, 0.0014314651489257812, -0.014923095703125, -0.011810302734375, 0.00212860107421875, 0.01128387451171875, 0.07513427734375, -0.0006647109985351562, -0.0087738037109375, -0.0275115966796875, -0.0469970703125, 0.037384033203125, -0.061004638671875, 0.0021152496337890625, 0.0034999847412109375, -0.0157623291015625, -0.050811767578125, -0.00872802734375, 0.0084991455078125, 0.016876220703125, -0.02911376953125, -0.0179443359375, -0.04864501953125, -0.021453857421875, 0.034423828125, -0.04217529296875, -0.0159912109375, 0.06256103515625, -0.00518035888671875, 0.0209503173828125, 0.0020008087158203125, 0.01085662841796875, -0.022796630859375, 0.0027523040771484375, -0.01129150390625, 0.00868988037109375, 0.006923675537109375, 0.016

In [6]:
context=[]

### Sending API Call to the Search Index

In [ ]:
import requests
import json


url = f"{service_endpoint}/indexes/{index_name}/docs/search?api-version=2023-11-01"

headers = {
    "Content-Type": "application/json",
    "api-key": key
}

body = {
    "count": True,
    "select": "document_title, content_text, locationMetadata, image_document_id",
    "vectorQueries": [
        {
            "vector": vectorised_user_query,
            "k": 10,
            "fields": "content_embedding",
            "kind": "vector"
        }
    ]
}

response = requests.post(url, headers=headers, data=json.dumps(body))
documents = response.json()['value']

for doc in documents:
    context.append(dict(
        {
            "document_title": doc['document_title'],
            "chunk": doc['content_text'],
            "score": doc['@search.score'],
            "locationMetadata": doc['locationMetadata'] if 'locationMetadata' in doc else None,
            "image_document_id": doc['image_document_id'] if 'image_document_id' in doc else None
        }
    ))

for doc in context:
    print(doc)

{'document_title': 'BMW-Group-Report-2022-en.pdf', 'chunk': 'Report Other Information107\n\nhttps://www.bmwiventures.com/\nhttps://www.bmwgroup.com/content/dam/grpw/websites/bmwgroup_com/ir/downloads/en/2023/bericht/BMW-Group-GRI-Index-2022-en.pdf#page=14\nhttps://www.bmwgroup.com/content/dam/grpw/websites/bmwgroup_com/ir/downloads/en/2023/bericht/BMW-Group-GRI-Index-2022-en.pdf#page=14\n\n\nBiodiversity\u2003\u2003  \u2003\u2003\nThe BMW\xa0Group also takes biodiversity into account at all its \nlocations. As part of this strategy, we endeavour to minimise any \npotentially negative impacts caused by its production and con-\nstruction activities. In addition, after analysing biodiversity with \nthe help of relevant biodiversity indicators, we identify targeted \nmeasures with a view to creating natural habitats at selected \nsites or when planning the construction of any new buildings. \nWith this strategy, we want not only to help preserve, but also to \nimprove on biodiversity in an

### Calling GPT Engine for Summarisation

In [8]:
system_prompt = f""""You are meant to behave as a RAG chatbot that derives its context from a database stored in Azure AI Search Solution.
please answer strictly from the context from the database provided and if you dont have an answer please politely say so. dont include any extra 
information that is not in the context and dont include links as well.
the context passed to you will be in the form of a pythonic list with each object in the list having the following structure:

{{
    "document_title": "the title of the document",
    "chunk": "the chunk of text from the document",
    "score": "the score of the match based on cosine similarity",
    "locationMetadata": "the location metadata if available, else None",
    "image_document_id": "the image document id if available, else None",
}}

the pythonic list contains best 10 matches to the user query based on cosine similarity of the embeddings of the user query and the review descriptions.
please structure your answers in a very professional manner and in such a way that the user does not get to know that its RAG working under the hood
and its as if they are talking to a human. """

user_prompt = f""" the user query is: {user_query}
the context is : {context}"""

chat_completions_response = azure_openai_client.chat.completions.create(
    model = os.getenv("AZURE_OPENAI_CHAT_COMPLETIONS_MODEL"),
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.7
)

print(chat_completions_response.choices[0].message.content)

BMW has undertaken a variety of sustainability initiatives that reflect its commitment to reducing environmental impact and promoting sustainable development. Here are some of the key areas of focus:

1. **Carbon Emissions Reduction**:
   - BMW has set science-based targets to achieve significant reductions in carbon emissions by 2030, aligned with the Paris Climate Agreement. These include:
     - An 80% reduction in carbon emissions (Scope 1 and 2) from its production sites and locations.
     - A 50% reduction in emissions during the vehicle use phase (Scope 3 downstream).
     - A 20% reduction in emissions in the supply chain (Scope 3 upstream).

2. **Renewable Energy and Resource Efficiency**:
   - BMW aims to minimize energy requirements at its facilities and increase the use of renewable energy. Efforts also include managing resource efficiency across its production network and reducing energy, potable water consumption, waste for disposal, and solvent usage by 25% per vehicle 